# 02 - YOLO fine-tuning on a Colab T4
Runtime > Change runtime type > T4 GPU.

In [ ]:
!git clone https://github.com/<YOUR_USER>/adpstack.git
%cd adpstack
!pip install -q -e ".[ml,export]"

In [ ]:
from google.colab import drive; drive.mount('/content/drive')
# Put the BDD100K 10K images + det_20 labels in Drive, then:
!mkdir -p data && ln -sfn /content/drive/MyDrive/bdd100k data/bdd100k

In [ ]:
!python -m adpstack.data.bdd100k_to_yolo --root data/bdd100k --out /content/bdd_yolo --subset 10000 --copy-images

In [ ]:
import wandb; wandb.login()

In [ ]:
!python -m adpstack.detection.train --set data=/content/bdd_yolo/dataset.yaml epochs=50 model=yolo11n.pt name=yolo11n

In [ ]:
!python -m adpstack.detection.train --set data=/content/bdd_yolo/dataset.yaml epochs=50 model=yolov8s.pt name=yolov8s   # ablation

In [ ]:
!python -m adpstack.detection.evaluate --weights runs/detect/yolo11n/weights/best.pt --data /content/bdd_yolo/dataset.yaml --labels-json data/bdd100k/labels/det_20/det_val.json --images-dir /content/bdd_yolo/images/val --device 0

In [ ]:
!cp runs/detect/yolo11n/weights/best.pt /content/drive/MyDrive/adp_models/yolo11n_best.pt